## 0. Local Infrastructure Setup

To ensure absolute compliance with enterprise data privacy standards and eliminate execution costs, this entire curriculum operates locally using **Ollama** running an optimized `llama3` instance. Run the following cells to initialize your local environment.

In [ ]:
# Step 1: Install system compression tools and dependencies
!sudo apt-get update && sudo apt-get install -y zstd

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,721 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,006 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-securi

In [ ]:
# Step 2: Download and install the core Ollama environment binary
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
# Step 3: Spin up the Ollama background host process
import os, subprocess, time
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(10)  # Verify backend execution completes system setup
print("Ollama engine successfully mounted on: http://localhost:11434")

Ollama engine successfully mounted on: http://localhost:11434


In [ ]:
# Step 4: Pull down the official local 'llama3' execution engine weights
!ollama pull llama3

In [ ]:
import requests
import json
import re

In [ ]:
OLLAMA_API_URL = "http://localhost:11434/api/generate"

def reasoned_chat(user_message):
    """
    Sends a message to the LLM with a strict structural guardrail,
    forcing it to 'think' inside specific tags before answering.
    """

    # 1. The System Prompt enforcing the reasoning structure
    system_prompt = """You are an intelligent retail support bot.
    You must NEVER jump straight to the answer. You must ALWAYS use this exact format:

    <thinking>
    Step 1: Analyze the user's request.
    Step 2: Calculate any numbers or verify policies step-by-step.
    Step 3: Draft the final conclusion.
    </thinking>
    <response>
    Write the final, friendly, polished reply to the user here.
    </response>
    """

    # Combine system prompt and user message
    full_prompt = f"{system_prompt}\n\nUser: {user_message}"

    payload = {
        "model": "llama3",
        "prompt": full_prompt,
        "stream": False,
        "options": {"temperature": 0.2} # Low temp for better logic
    }

    # 2. Call the Model
    print("AI is thinking...")
    response = requests.post(OLLAMA_API_URL, json=payload).json()["response"]

    # 3. Parse the output to separate the 'brain' from the 'mouth'
    try:
        # Use regex to extract the text inside the tags
        thinking_process = re.search(r'<thinking>(.*?)</thinking>', response, re.DOTALL).group(1).strip()
        final_answer = re.search(r'<response>(.*?)</response>', response, re.DOTALL).group(1).strip()
    except AttributeError:
        # Fallback just in case the model breaks the format guardrail
        thinking_process = "Failed to parse thinking."
        final_answer = response

    # 4. Display the results
    print("INTERNAL REASONING (Hidden from User):")
    print(thinking_process)
    print("FINAL CHATBOT RESPONSE (Shown to User):")
    print(final_answer)

In [ ]:
# --- Test the Chatbot ---
complex_user_query = "I bought 3 shirts at $25 each. I have a 20% off coupon, but shipping is $10. What is my total? And if my total is over $60, do I get free shipping?"

reasoned_chat(complex_user_query)

AI is thinking...
INTERNAL REASONING (Hidden from User):
Failed to parse thinking.
FINAL CHATBOT RESPONSE (Shown to User):
<thinking>

Step 1: Analyze the user's request.
Okay, let's break it down! The user bought 3 shirts at $25 each, which means the total cost of the shirts is... (drumroll please)... $75!

Step 2: Calculate any numbers or verify policies step-by-step.
Now, let's apply that 20% off coupon. To calculate the discount, I'll multiply the total by 0.20 (which is equivalent to 20%). That gives me... $15 in savings! So, subtracting that from the original total, my new subtotal is... $60!

Step 3: Draft the final conclusion.
Now, let's talk about shipping. The user mentioned it costs $10, so I'll add that to their subtotal. My total comes out to be... $70!

As for free shipping, the user asked if their total would exceed $60. Since my calculated total is indeed above that threshold ($70), they do qualify for free shipping! Woohoo!

</thinking>

<response>
Hi there! Based on y